# Build the Inspection-Level and Restaurant-Level History Tables

## Purpose

This notebook creates two analysis-ready tables from the team's prepared dataset:

1. **Inspection-level table:** one row per unique inspection, identified by `CAMIS`, `INSPECTION DATE`, and `INSPECTION TYPE`.
2. **Restaurant-level history table:** one row per `CAMIS`, summarizing each restaurant's inspection history.

The prepared file contains one row per violation, so multiple rows can belong to the same inspection. The notebook groups those violation rows into an inspection record and calculates violation counts before summarizing history by restaurant.

## Data decisions
- Exclude rows marked `AWAITING_INSPECTION == True` from inspection counts and inspection history.
- Keep restaurants with no real inspections in the restaurant-level table. They will have zero inspections and missing score/date summaries.
- Preserve missing scores and grades. Do not impute them.
- Do not remove high scores as outliers in this table-building step.

## Output files

The notebook writes `data/inspection_level.csv` and `data/restaurant_history.csv`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

## 1. Load the prepared dataset

In [5]:
# Keep identifier columns as text so leading zeros are not lost.
id_dtypes = {
    "ZIPCODE": "string",
    "Community Board": "string",
    "Council District": "string",
    "Census Tract": "string",
    "BIN": "string",
    "BBL": "string",
}

df_prepared = pd.read_csv(
    "data/restaurants_prepared.csv",
    dtype=id_dtypes,
    parse_dates=["INSPECTION DATE", "GRADE DATE", "RECORD DATE"],
    low_memory=False,
)

print(df_prepared.shape)

(295843, 32)


In [6]:
df_prepared.head(10)

,SOURCE_ROW_ID,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,...,Council District,Census Tract,BIN,BBL,NTA,Location,AWAITING_INSPECTION,BORO_UNKNOWN,ZERO_COORDINATES,BBL_INCOMPLETE
0,1,50181368,FROYOLICIOUS,Staten Island,2955,VETERANS ROAD WEST,10309,9177836601,NaN,1900-01-01,...,51,022600,5150131,5075110001,SI11,POINT (-74.23264118855 40.527800352215),True,False,False,False
1,2,41395379,CITI FIELD STAND 114,Queens,000,126th & ROOSEVELT AVENUE,<NA>,7185958100,American,2024-09-20,...,<NA>,<NA>,<NA>,<NA>,NaN,NaN,False,False,True,True
2,3,50169220,HERITAGE DELI,Queens,56-27,58 STREET,11378,7187915831,NaN,1900-01-01,...,26,021900,4059724,4026640008,QN31,POINT (-73.912449613268 40.724601345342),True,False,False,False
3,4,50148845,MAKE MY PASTRY,Queens,40-05,SKILLMAN AVENUE,11104,3479385678,NaN,1900-01-01,...,26,017100,4002375,4001830292,QN31,POINT (-73.923416505935 40.747195864681),True,False,False,False
4,5,50162734,IQ FOODS GROUP LLC,Manhattan,1,HERALD SQUARE,10001,3472440804,NaN,1900-01-01,...,04,010900,1089436,1008100045,MN17,POINT (-73.987541021756 40.750541907818),True,False,False,False
5,6,50179812,KAMPOS TAVERNA,Brooklyn,8413,3 AVENUE,11209,3478570516,NaN,1900-01-01,...,47,006200,3152742,3060250005,BK31,POINT (-74.030479756047 40.624841632068),True,False,False,False
6,7,50189901,MARCO VACCHI,Manhattan,646,9 AVENUE,10036,3477811121,NaN,1900-01-01,...,03,012100,1025004,1010360003,MN15,POINT (-73.991051426489 40.760571586789),True,False,False,False
7,8,41194745,GOLDEN PALACE GOURMET,Queens,14009,CHERRRY AVENUE,<NA>,7188864383,Chinese,2024-12-09,...,<NA>,<NA>,<NA>,<NA>,NaN,NaN,False,False,True,True
8,9,50168503,LA QUINTA INN AND SUITES CENTRAL PARK,Manhattan,31,WEST 71 STREET,10023,5169159068,NaN,1900-01-01,...,06,015700,1028614,1011240018,MN14,POINT (-73.977982557122 40.776179253854),True,False,False,False
9,10,50178739,WOOPS MACARONS,Manhattan,151,WEST 34 STREET,10001,9175876070,NaN,1900-01-01,...,03,010900,1015219,1008100001,MN17,POINT (-73.989479170444 40.750509164767),True,False,False,False


## 2. Prepare the real inspection records

Before creating either table, confirm that the columns used below are present and that the inspection-status flag is available.

In [7]:
required_columns = [
    "SOURCE_ROW_ID", "CAMIS", "DBA", "BORO", "CUISINE DESCRIPTION",
    "INSPECTION DATE", "INSPECTION TYPE", "ACTION", "SCORE", "GRADE",
    "GRADE DATE", "VIOLATION CODE", "VIOLATION DESCRIPTION", "CRITICAL FLAG",
    "AWAITING_INSPECTION",
]

missing_columns = [col for col in required_columns if col not in df_prepared.columns]
if missing_columns:
    raise ValueError(f"The prepared dataset is missing required columns: {missing_columns}")

In [8]:
if df_prepared["CAMIS"].isna().any():
    raise ValueError("CAMIS contains missing values. A restaurant-level table cannot be built safely.")

if df_prepared["AWAITING_INSPECTION"].isna().any():
    raise ValueError("AWAITING_INSPECTION contains missing values. Review the prepared dataset before continuing.")

print("Rows:", len(df_prepared))
print("Columns:", df_prepared.shape[1])
print("Unique restaurants:", df_prepared["CAMIS"].nunique())
print("Rows awaiting inspection:", int(df_prepared["AWAITING_INSPECTION"].sum()))
print("Rows with a real inspection:", int((~df_prepared["AWAITING_INSPECTION"]).sum()))
print("Unique SOURCE_ROW_ID values:", df_prepared["SOURCE_ROW_ID"].nunique())

Rows: 295843
Columns: 32
Unique restaurants: 31366
Rows awaiting inspection: 3714
Rows with a real inspection: 292129
Unique SOURCE_ROW_ID values: 295843


# Inspection Level Table
---
## 3. Separate Real Inspection Rows

`AWAITING_INSPECTION` identifies placeholder records dated January 1, 1900. These rows represent restaurants that have not yet had an inspection, so they should remain available for the restaurant-level table but should not be counted as inspections.

In [9]:
df_real = df_prepared.loc[~df_prepared["AWAITING_INSPECTION"]].copy()

inspection_key = ["CAMIS", "INSPECTION DATE", "INSPECTION TYPE"]

# Every real inspection needs all three parts of the inspection key.
missing_key_values = df_real[inspection_key].isna().sum()
display(missing_key_values.to_frame(name="missing_values"))

if df_real[inspection_key].isna().any().any():
    raise ValueError(
        "At least one real inspection is missing part of its inspection key. "
        "Review these records before grouping."
    )

print("Rows belonging to real inspections:", len(df_real))
print("Restaurants with at least one real inspection:", df_real["CAMIS"].nunique())

,missing_values
CAMIS,0
INSPECTION DATE,0
INSPECTION TYPE,0


Rows belonging to real inspections: 292129
Restaurants with at least one real inspection: 27652


## 4. Check that inspection-level fields are consistent

The team previously checked that score, grade, grade date, action, restaurant name, borough, cuisine, and ZIP code are consistent within each inspection. I repeat that check here because the inspection-level table keeps one representative row for those fields.

In [10]:
consistency_columns = [
    "SCORE", "GRADE", "GRADE DATE", "ACTION", "DBA", "BORO",
    "CUISINE DESCRIPTION", "ZIPCODE",
]

grouped_real = df_real.groupby(inspection_key, dropna=False, sort=False)
consistency_check = pd.DataFrame(
    {
        "inspections_with_different_values": [
            int((grouped_real[col].nunique(dropna=False) > 1).sum())
            for col in consistency_columns
        ]
    },
    index=consistency_columns,
)

display(consistency_check)

if consistency_check["inspections_with_different_values"].sum() > 0:
    print(
        "WARNING: Some inspection-level fields differ within an inspection. "
        "Review the consistency table before relying on the representative rows."
    )
else:
    print("All checked fields are consistent within each inspection.")

,inspections_with_different_values
SCORE,0
GRADE,0
GRADE DATE,0
ACTION,0
DBA,0
BORO,0
CUISINE DESCRIPTION,0
ZIPCODE,0


All checked fields are consistent within each inspection.


## 5. Build the inspection-level table

The source data has one row per violation, not one row per inspection. I first calculate violation counts for each inspection, then keep one representative row for the inspection's shared details.
- `violation_record_count` counts all source rows grouped into the inspection.
- `total_violation_count` counts rows with a non-missing violation code. This avoids counting a no-violation record as a violation.
- `missing_violation_code_row_count` keeps track of rows where no violation code was recorded.
- Critical and non-critical counts include only rows with a violation code.

The violation-specific columns (`VIOLATION CODE`, `VIOLATION DESCRIPTION`, and `CRITICAL FLAG`) are summarized rather than copied from one arbitrary violation row.

In [14]:
# Create temporary indicators so each violation category can be counted clearly.
df_real["_has_violation_code"] = df_real["VIOLATION CODE"].notna()
df_real["_is_critical_violation"] = (
    df_real["_has_violation_code"] & df_real["CRITICAL FLAG"].eq("Critical")
)
df_real["_is_non_critical_violation"] = (
    df_real["_has_violation_code"] & df_real["CRITICAL FLAG"].eq("Not Critical")
)

inspection_counts = (
    df_real.groupby(inspection_key, dropna=False, sort=False)
    .agg(
        violation_record_count=("SOURCE_ROW_ID", "size"),
        total_violation_count=("_has_violation_code", "sum"),
        missing_violation_code_row_count=("_has_violation_code", lambda s: (~s).sum()),
        critical_violation_count=("_is_critical_violation", "sum"),
        non_critical_violation_count=("_is_non_critical_violation", "sum"),
        not_applicable_flag_record_count=(
            "CRITICAL FLAG", lambda s: s.eq("Not Applicable").sum()
        ),
        unique_violation_code_count=("VIOLATION CODE", "nunique"),
        violation_description_record_count=("VIOLATION DESCRIPTION", "count"),
    )
    .reset_index()
)

In [15]:
# Keep the first source row for each inspection as the representative record.
# The SOURCE_ROW_ID is renamed to make its role clear in the aggregated table.
inspection_base = (
    df_real.sort_values("SOURCE_ROW_ID", kind="stable")
    .drop_duplicates(subset=inspection_key, keep="first")
    .drop(columns=["VIOLATION CODE", "VIOLATION DESCRIPTION", "CRITICAL FLAG"])
    .rename(columns={"SOURCE_ROW_ID": "REPRESENTATIVE_SOURCE_ROW_ID"})
)

inspection_level = inspection_base.merge(
    inspection_counts,
    on=inspection_key,
    how="left",
    validate="one_to_one",
)

In [20]:
# Add score bands based on the NYC score ranges. Missing scores stay missing.
inspection_level["score_band"] = pd.Series(
    pd.NA, index=inspection_level.index, dtype="string"
)
score = inspection_level["SCORE"]
inspection_level.loc[score.between(0, 13), "score_band"] = "A range (0-13)"
inspection_level.loc[score.between(14, 27), "score_band"] = "B range (14-27)"
inspection_level.loc[score.ge(28), "score_band"] = "C range (28+)"

inspection_level["critical_violation_pct"] = np.where(
    inspection_level["total_violation_count"] > 0,
    inspection_level["critical_violation_count"]
    / inspection_level["total_violation_count"] * 100,
    np.nan,
)

In [22]:
inspection_level = inspection_level.sort_values(
    ["CAMIS", "INSPECTION DATE", "INSPECTION TYPE"], kind="stable"
).reset_index(drop=True)

inspection_level.head(10)

,REPRESENTATIVE_SOURCE_ROW_ID,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,...,violation_record_count,total_violation_count,missing_violation_code_row_count,critical_violation_count,non_critical_violation_count,not_applicable_flag_record_count,unique_violation_code_count,violation_description_record_count,score_band,critical_violation_pct
0,172870,30075445,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462,7188924968,Bakery Products/Desserts,2023-08-01,...,3,3,0,2,1,0,3,3,C range (28+),66.666667
1,210601,30075445,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462,7188924968,Bakery Products/Desserts,2023-08-22,...,3,3,0,1,2,0,3,3,A range (0-13),33.333333
2,9888,30075445,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462,7188924968,Bakery Products/Desserts,2024-11-08,...,3,3,0,1,2,0,3,3,A range (0-13),33.333333
3,147982,30075445,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462,7188924968,Bakery Products/Desserts,2026-02-27,...,2,2,0,1,1,0,2,2,A range (0-13),50.000000
4,45216,30191841,D.J. REYNOLDS,Manhattan,351,WEST 57 STREET,10019,2122452912,Irish,2024-11-20,...,6,6,0,2,4,0,6,6,B range (14-27),33.333333
5,60973,30191841,D.J. REYNOLDS,Manhattan,351,WEST 57 STREET,10019,2122452912,Irish,2025-02-20,...,2,2,0,2,0,0,2,2,A range (0-13),100.000000
6,169346,30191841,D.J. REYNOLDS,Manhattan,351,WEST 57 STREET,10019,2122452912,Irish,2026-07-30,...,3,3,0,2,1,0,3,3,A range (0-13),66.666667
7,10696,40356483,WILKEN'S FINE FOOD,Brooklyn,7114,AVENUE U,11234,7184443838,Sandwiches,2023-11-16,...,6,6,0,3,3,0,6,6,C range (28+),50.000000
8,59051,40356483,WILKEN'S FINE FOOD,Brooklyn,7114,AVENUE U,11234,7184443838,Sandwiches,2024-04-23,...,5,5,0,3,2,0,5,5,B range (14-27),60.000000
9,195052,40356483,WILKEN'S FINE FOOD,Brooklyn,7114,AVENUE U,11234,7184443838,Sandwiches,2025-01-06,...,2,2,0,2,0,0,2,2,A range (0-13),100.000000


## 6. Validate the inspection-level table

The checks below make sure that each inspection appears once and that the violation-row totals reconcile to the real-inspection rows in the prepared dataset.

In [23]:
print("Inspection-level table shape:", inspection_level.shape)
print("Unique inspection keys:", inspection_level[inspection_key].drop_duplicates().shape[0])
print("Duplicate inspection keys:", inspection_level.duplicated(subset=inspection_key).sum())
print("Inspection rows represented:", inspection_level["violation_record_count"].sum())
print("Real source rows:", len(df_real))
print("Violation codes counted:", inspection_level["total_violation_count"].sum())
print("Non-missing violation codes in source:", df_real["VIOLATION CODE"].notna().sum())
print("Inspections with a score:", inspection_level["SCORE"].notna().sum())
print("Inspections without a score:", inspection_level["SCORE"].isna().sum())

assert inspection_level[inspection_key].duplicated().sum() == 0
assert inspection_level["violation_record_count"].sum() == len(df_real)
assert inspection_level["total_violation_count"].sum() == df_real["VIOLATION CODE"].notna().sum()

print("\nInspection-level validation passed.")

Inspection-level table shape: (93292, 42)
Unique inspection keys: 93292
Duplicate inspection keys: 0
Inspection rows represented: 292129
Real source rows: 292129
Violation codes counted: 289399
Non-missing violation codes in source: 289399
Inspections with a score: 82956
Inspections without a score: 10336

Inspection-level validation passed.


# Restaurant Level Table
---

## 7. Build the restaurant profile
The restaurant-level table must include all restaurants in the prepared dataset, including restaurants that have not yet been inspected. I create one profile row per `CAMIS` using the latest non-missing value available for each restaurant detail. This helps retain an available address or cuisine value if the latest source row happens to be missing that field.

In [24]:
restaurant_profile_columns = [
    "DBA", "BORO", "BUILDING", "STREET", "ZIPCODE", "PHONE",
    "CUISINE DESCRIPTION", "Latitude", "Longitude", "Community Board",
    "Council District", "Census Tract", "BIN", "BBL", "NTA", "Location",
    "BORO_UNKNOWN", "ZERO_COORDINATES", "BBL_INCOMPLETE",
]

# Sort chronologically. GroupBy.last() takes the latest non-missing value per field.
restaurant_profile = (
    df_prepared.sort_values(
        ["INSPECTION DATE", "SOURCE_ROW_ID"], kind="stable"
    )
    .groupby("CAMIS", as_index=False, sort=False)[restaurant_profile_columns]
    .last()
)

print("Restaurant profile rows:", len(restaurant_profile))
print("Unique CAMIS values in profile:", restaurant_profile["CAMIS"].nunique())
display(restaurant_profile.head())

Restaurant profile rows: 31366
Unique CAMIS values in profile: 31366


,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location,BORO_UNKNOWN,ZERO_COORDINATES,BBL_INCOMPLETE
0,50181368,FROYOLICIOUS,Staten Island,2955,VETERANS ROAD WEST,10309,9177836601,None,40.527800,-74.232641,503,51,022600,5150131,5075110001,SI11,POINT (-74.23264118855 40.527800352215),False,False,False
1,50169220,HERITAGE DELI,Queens,56-27,58 STREET,11378,7187915831,None,40.724601,-73.912450,402,26,021900,4059724,4026640008,QN31,POINT (-73.912449613268 40.724601345342),False,False,False
2,50148845,MAKE MY PASTRY,Queens,40-05,SKILLMAN AVENUE,11104,3479385678,None,40.747196,-73.923417,402,26,017100,4002375,4001830292,QN31,POINT (-73.923416505935 40.747195864681),False,False,False
3,50162734,IQ FOODS GROUP LLC,Manhattan,1,HERALD SQUARE,10001,3472440804,None,40.750542,-73.987541,105,04,010900,1089436,1008100045,MN17,POINT (-73.987541021756 40.750541907818),False,False,False
4,50179812,KAMPOS TAVERNA,Brooklyn,8413,3 AVENUE,11209,3478570516,None,40.624842,-74.030480,310,47,006200,3152742,3060250005,BK31,POINT (-74.030479756047 40.624841632068),False,False,False


## 8. Summarize inspection history by restaurant
All history metrics are calculated from `inspection_level`, which already has one row per inspection. This is important: calculating the number of inspections directly from the violation-level source would overcount inspections that had multiple violations.

Score averages and score-band percentages use only inspections with a recorded score. Missing scores and grades remain missing; they are not filled in.

In [25]:
restaurant_summary = (
    inspection_level.groupby("CAMIS", as_index=False, sort=False)
    .agg(
        total_inspections=("INSPECTION DATE", "size"),
        first_inspection_date=("INSPECTION DATE", "min"),
        most_recent_inspection_date=("INSPECTION DATE", "max"),
        inspections_with_score=("SCORE", "count"),
        average_score=("SCORE", "mean"),
        median_score=("SCORE", "median"),
        lowest_score=("SCORE", "min"),
        highest_score=("SCORE", "max"),
        total_violation_count=("total_violation_count", "sum"),
        critical_violation_count=("critical_violation_count", "sum"),
        non_critical_violation_count=("non_critical_violation_count", "sum"),
        inspections_with_critical_violations=(
            "critical_violation_count", lambda s: s.gt(0).sum()
        ),
        a_range_inspections=("score_band", lambda s: s.eq("A range (0-13)").sum()),
        b_range_inspections=("score_band", lambda s: s.eq("B range (14-27)").sum()),
        c_range_inspections=("score_band", lambda s: s.eq("C range (28+)").sum()),
        grade_a_inspections=("GRADE", lambda s: s.eq("A").sum()),
        grade_b_inspections=("GRADE", lambda s: s.eq("B").sum()),
        grade_c_inspections=("GRADE", lambda s: s.eq("C").sum()),
        grade_n_status_inspections=("GRADE", lambda s: s.eq("N").sum()),
        grade_p_status_inspections=("GRADE", lambda s: s.eq("P").sum()),
        grade_z_status_inspections=("GRADE", lambda s: s.eq("Z").sum()),
    )
)

print("Restaurants represented in inspection history:", len(restaurant_summary))
display(restaurant_summary.head())

Restaurants represented in inspection history: 27652


,CAMIS,total_inspections,first_inspection_date,most_recent_inspection_date,inspections_with_score,average_score,median_score,lowest_score,highest_score,total_violation_count,...,inspections_with_critical_violations,a_range_inspections,b_range_inspections,c_range_inspections,grade_a_inspections,grade_b_inspections,grade_c_inspections,grade_n_status_inspections,grade_p_status_inspections,grade_z_status_inspections
0,30075445,4,2023-08-01,2026-02-27,4,16.750000,11.0,7.0,38.0,11,...,4,3,0,1,3,0,0,0,0,0
1,30191841,3,2024-11-20,2026-07-30,3,15.666667,13.0,10.0,24.0,11,...,3,2,1,0,2,0,0,0,0,0
2,40356483,5,2023-11-16,2026-06-10,4,21.500000,19.5,12.0,35.0,18,...,4,2,1,1,2,1,0,0,0,0
3,40356731,3,2024-04-08,2026-06-25,3,9.666667,12.0,5.0,12.0,8,...,2,3,0,0,3,0,0,0,0,0
4,40357217,2,2024-05-21,2025-10-20,2,12.000000,12.0,12.0,12.0,4,...,2,2,0,0,2,0,0,0,0,0


## 9. Add each restaurant's latest and scored inspection details

The latest inspection may not have a score yet. Therefore, the notebook keeps the latest inspection's score separately from the latest inspection that actually has a score. It also records the first scored inspection so a simple first-to-latest scored comparison can be calculated.

In [26]:
inspection_sorted = inspection_level.sort_values(
    ["CAMIS", "INSPECTION DATE", "REPRESENTATIVE_SOURCE_ROW_ID"],
    kind="stable",
)

latest_inspection = (
    inspection_sorted.drop_duplicates("CAMIS", keep="last")
    [[
        "CAMIS", "REPRESENTATIVE_SOURCE_ROW_ID", "SCORE", "GRADE",
        "INSPECTION TYPE", "ACTION",
    ]]
    .rename(columns={
        "REPRESENTATIVE_SOURCE_ROW_ID": "latest_inspection_source_row_id",
        "SCORE": "latest_inspection_score",
        "GRADE": "latest_inspection_grade",
        "INSPECTION TYPE": "latest_inspection_type",
        "ACTION": "latest_inspection_action",
    })
)

scored_inspections = inspection_sorted.loc[inspection_sorted["SCORE"].notna()].copy()

first_scored_inspection = (
    scored_inspections.drop_duplicates("CAMIS", keep="first")
    [["CAMIS", "INSPECTION DATE", "SCORE"]]
    .rename(columns={
        "INSPECTION DATE": "first_scored_inspection_date",
        "SCORE": "first_scored_score",
    })
)

latest_scored_inspection = (
    scored_inspections.drop_duplicates("CAMIS", keep="last")
    [["CAMIS", "INSPECTION DATE", "SCORE"]]
    .rename(columns={
        "INSPECTION DATE": "most_recent_scored_inspection_date",
        "SCORE": "most_recent_scored_score",
    })
)

print("Restaurants with at least one scored inspection:", len(latest_scored_inspection))

Restaurants with at least one scored inspection: 27647


## 10. Assemble the restaurant-level history table
Start with the full restaurant profile and merge the inspection summaries onto it. Because this is a left join from all restaurants, restaurants with no real inspections remain in the output.

In [27]:
restaurant_history = (
    restaurant_profile
    .merge(restaurant_summary, on="CAMIS", how="left", validate="one_to_one")
    .merge(latest_inspection, on="CAMIS", how="left", validate="one_to_one")
    .merge(first_scored_inspection, on="CAMIS", how="left", validate="one_to_one")
    .merge(latest_scored_inspection, on="CAMIS", how="left", validate="one_to_one")
)

# Fill only count fields with zero. Score and date summaries remain missing
# for restaurants without an inspection or without a recorded score.
count_columns = [
    "total_inspections", "inspections_with_score", "total_violation_count",
    "critical_violation_count", "non_critical_violation_count",
    "inspections_with_critical_violations", "a_range_inspections",
    "b_range_inspections", "c_range_inspections", "grade_a_inspections",
    "grade_b_inspections", "grade_c_inspections", "grade_n_status_inspections",
    "grade_p_status_inspections", "grade_z_status_inspections",
]

for col in count_columns:
    restaurant_history[col] = restaurant_history[col].fillna(0).astype("int64")

restaurant_history["inspections_without_score"] = (
    restaurant_history["total_inspections"]
    - restaurant_history["inspections_with_score"]
)

# Calculate percentages only when the denominator is greater than zero.
for band in ["a", "b", "c"]:
    restaurant_history[f"{band}_range_score_pct"] = np.where(
        restaurant_history["inspections_with_score"] > 0,
        restaurant_history[f"{band}_range_inspections"]
        / restaurant_history["inspections_with_score"] * 100,
        np.nan,
    )

restaurant_history["critical_inspection_pct"] = np.where(
    restaurant_history["total_inspections"] > 0,
    restaurant_history["inspections_with_critical_violations"]
    / restaurant_history["total_inspections"] * 100,
    np.nan,
)

restaurant_history["score_change_first_to_latest"] = (
    restaurant_history["most_recent_scored_score"]
    - restaurant_history["first_scored_score"]
)

restaurant_history["has_real_inspection"] = restaurant_history["total_inspections"] > 0
restaurant_history["awaiting_first_inspection"] = restaurant_history["total_inspections"] == 0

restaurant_history = restaurant_history.sort_values("CAMIS", kind="stable").reset_index(drop=True)

display(restaurant_history.head(10))

,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,Latitude,Longitude,...,most_recent_scored_inspection_date,most_recent_scored_score,inspections_without_score,a_range_score_pct,b_range_score_pct,c_range_score_pct,critical_inspection_pct,score_change_first_to_latest,has_real_inspection,awaiting_first_inspection
0,30075445,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462,7188924968,Bakery Products/Desserts,40.848231,-73.855972,...,2026-02-27,7.0,0,75.000000,0.000000,25.000000,100.000000,-31.0,True,False
1,30191841,D.J. REYNOLDS,Manhattan,351,WEST 57 STREET,10019,2122452912,Irish,40.767326,-73.984310,...,2026-07-30,13.0,0,66.666667,33.333333,0.000000,100.000000,-11.0,True,False
2,40356078,"I. & L. DELICACIES,INC.",Manhattan,1500-2N,D AVE.,<NA>,None,None,NaN,NaN,...,NaT,NaN,0,NaN,NaN,NaN,NaN,NaN,False,True
3,40356483,WILKEN'S FINE FOOD,Brooklyn,7114,AVENUE U,11234,7184443838,Sandwiches,40.620112,-73.906989,...,2026-06-10,12.0,1,50.000000,25.000000,25.000000,80.000000,-23.0,True,False
4,40356731,TASTE THE TROPICS ICE CREAM,Brooklyn,1839,NOSTRAND AVENUE,11226,7188560821,Frozen Desserts,40.640795,-73.948488,...,2026-06-25,5.0,0,100.000000,0.000000,0.000000,66.666667,-7.0,True,False
5,40357217,ASIA PLAZA CAFÉ,Bronx,2300,SOUTHERN BOULEVARD,10460,7187411426,American,40.850537,-73.882455,...,2025-10-20,12.0,0,100.000000,0.000000,0.000000,100.000000,0.0,True,False
6,40357297,O & G GROCERIES,Manhattan,112,GREENWICH AVENUE,10011,None,None,40.738474,-74.002226,...,NaT,NaN,0,NaN,NaN,NaN,NaN,NaN,False,True
7,40359480,1 EAST 66TH STREET KITCHEN,Manhattan,1,EAST 66 STREET,10065,2128793900,American,40.768547,-73.969581,...,2025-12-02,10.0,0,100.000000,0.000000,0.000000,100.000000,0.0,True,False
8,40359483,ANTHONY ASPERTI,Staten Island,30,GARIBALDI AVE,10306,None,None,40.562283,-74.101643,...,NaT,NaN,0,NaN,NaN,NaN,NaN,NaN,False,True
9,40359705,NATHAN'S FAMOUS,Brooklyn,1310,SURF AVENUE,11224,7183332202,Hotdogs,40.575537,-73.981652,...,2026-05-07,12.0,0,66.666667,0.000000,33.333333,66.666667,-39.0,True,False


## 11. Validate both tables
These checks verify that:
- The inspection-level table has one row per unique inspection key.
- The restaurant-level table has one row per restaurant in the prepared dataset.
- Inspection counts and violation counts reconcile between the two tables.
- Restaurants awaiting their first inspection are retained with zero inspections and missing score/date summaries.

In [28]:
total_restaurants = df_prepared["CAMIS"].nunique()
restaurants_with_real_inspections = df_real["CAMIS"].nunique()
restaurants_awaiting = df_prepared.loc[
    df_prepared["AWAITING_INSPECTION"], "CAMIS"
].nunique()

validation_results = pd.Series(
    {
        "Prepared dataset unique restaurants": total_restaurants,
        "Inspection-level rows": len(inspection_level),
        "Inspection-level unique keys": inspection_level[inspection_key].drop_duplicates().shape[0],
        "Restaurant-level rows": len(restaurant_history),
        "Restaurant-level unique CAMIS": restaurant_history["CAMIS"].nunique(),
        "Restaurants with real inspections": restaurants_with_real_inspections,
        "Restaurants awaiting first inspection": int(restaurant_history["awaiting_first_inspection"].sum()),
        "Total inspections in restaurant history": int(restaurant_history["total_inspections"].sum()),
        "Total violations in inspection table": int(inspection_level["total_violation_count"].sum()),
        "Total violations in restaurant table": int(restaurant_history["total_violation_count"].sum()),
    },
    name="count",
)

display(validation_results.to_frame())

assert inspection_level[inspection_key].duplicated().sum() == 0
assert restaurant_history["CAMIS"].is_unique
assert len(restaurant_history) == total_restaurants
assert restaurant_history["total_inspections"].sum() == len(inspection_level)
assert restaurant_history["total_violation_count"].sum() == inspection_level["total_violation_count"].sum()
assert restaurant_history["awaiting_first_inspection"].sum() == restaurants_awaiting

# These are the team's previously documented counts. They are printed as a
# comparison rather than hard-coded as assertions in case the source snapshot changes.
print("\nComparison with the team's documented checkpoint:")
print("Expected unique restaurants: 31,366 | Actual:", total_restaurants)
print("Expected real inspections: 93,292 | Actual:", len(inspection_level))
print("Expected restaurants with real inspections: 27,652 | Actual:", restaurants_with_real_inspections)
print("Expected restaurants awaiting first inspection: 3,714 | Actual:", restaurants_awaiting)
print("\nAll internal validation checks passed.")

,count
Prepared dataset unique restaurants,31366
Inspection-level rows,93292
Inspection-level unique keys,93292
Restaurant-level rows,31366
Restaurant-level unique CAMIS,31366
Restaurants with real inspections,27652
Restaurants awaiting first inspection,3714
Total inspections in restaurant history,93292
Total violations in inspection table,289399
Total violations in restaurant table,289399



Comparison with the team's documented checkpoint:
Expected unique restaurants: 31,366 | Actual: 31366
Expected real inspections: 93,292 | Actual: 93292
Expected restaurants with real inspections: 27,652 | Actual: 27652
Expected restaurants awaiting first inspection: 3,714 | Actual: 3714

All internal validation checks passed.


## 12. Save the two output tables
Both files are saved in the existing data/ folder.

In [29]:
inspection_level.to_csv(
    "data/inspection_level.csv",
    index=False
)

restaurant_history.to_csv(
    "data/restaurant_history.csv",
    index=False
)

print("Inspection-level shape:", inspection_level.shape)
print("Restaurant-level shape:", restaurant_history.shape)

Inspection-level shape: (93292, 42)
Restaurant-level shape: (31366, 58)


## Handoff Notes

The two tables are ready for the EDA portion of the project:
- data/inspection_level.csv is the table to use when the unit of analysis is an individual inspection.
- data/restaurant_history.csv is the table to use when the unit of analysis is a restaurant and its overall inspection history.
The inspection-level table includes one representative source-row ID (`REPRESENTATIVE_SOURCE_ROW_ID`) for traceability. The violation counts summarize all source rows belonging to that inspection; the representative ID is not intended to identify every violation row.

`total_violation_count` counts records with a non-missing VIOLATION CODE. `violation_record_count` counts all source rows belonging to the inspection, and `missing_violation_code_row_count` records rows without a violation code. This distinction is useful because some inspection records have no violation code recorded.

Score bands are based on the team's stated ranges: 0–13, 14–27, and 28 or more. High scores are retained. A positive `score_change_first_to_latest` means the restaurant's most recent scored inspection had a higher (worse) score than its first scored inspection; it is a simple comparison, not a model of the trend over time.